# Advanced Problems with Solutions: Random Choices in Python

This notebook contains advanced practice problems about `random.choice`, `random.choices`, sampling with replacement, weighted choices, cumulative weights, frequency analysis, reproducibility, and randomized benchmarking.

## Learning goals

By the end, you should be able to:

- Choose random elements from sequences using `random.choice`.
- Generate multiple random choices using `random.choices`.
- Explain sampling with replacement.
- Use `weights` and `cum_weights` correctly.
- Analyze empirical frequencies.
- Compare observed frequencies with theoretical probabilities.
- Build reproducible random experiments.
- Use weighted data generation for realistic benchmarks.

In [1]:
import random
from collections import Counter, namedtuple
from time import perf_counter
from statistics import mean

## Problem 1: Replace Manual Index Selection with `choice`

A beginner writes the following code to select a random item from a list:

```python
items = [10, 20, 30, 40, 50, 60]
index = random.randrange(len(items))
item = items[index]
```

Although this works, it is not the cleanest approach.

### Task

1. Write a function `pick_item(items, seed=None)` that uses `random.choice`.
2. Make the function reproducible.
3. Prove that the same seed produces the same selected item.
4. Raise a helpful error if the sequence is empty.

### Solution

In [2]:
def pick_item(items, seed=None):
    if len(items) == 0:
        raise ValueError("Cannot choose from an empty sequence.")
    
    rng = random.Random(seed)
    return rng.choice(items)


items = [10, 20, 30, 40, 50, 60]

first = pick_item(items, seed=0)
second = pick_item(items, seed=0)

print(first)
print(second)

assert first == second

try:
    pick_item([], seed=0)
except ValueError as ex:
    print("Error:", ex)

40
40
Error: Cannot choose from an empty sequence.


### Explanation

`random.choice(sequence)` directly expresses the intention: choose one random element from a non-empty sequence. Using `random.Random(seed)` avoids modifying the global random state.

## Problem 2: Sampling With Replacement

`random.choices` selects elements with replacement. That means the same element can appear more than once in a single result.

### Task

Use `random.choices` to generate 20 samples of size 5 from the population `['a', 'b', 'c']`.

Then count how many of those 20 samples contain at least one repeated element.

### Solution

In [3]:
def has_repeated_element(sample):
    return len(sample) != len(set(sample))


rng = random.Random(0)
population = ['a', 'b', 'c']

samples = [rng.choices(population, k=5) for _ in range(20)]
num_with_repeats = sum(has_repeated_element(sample) for sample in samples)

for sample in samples:
    print(sample)

print("\nSamples with repeated elements:", num_with_repeats)

assert num_with_repeats == 20

['c', 'c', 'b', 'a', 'b']
['b', 'c', 'a', 'b', 'b']
['c', 'b', 'a', 'c', 'b']
['a', 'c', 'c', 'c', 'c']
['a', 'c', 'c', 'c', 'b']
['a', 'b', 'b', 'c', 'c']
['b', 'c', 'a', 'c', 'b']
['a', 'c', 'b', 'c', 'c']
['a', 'b', 'c', 'a', 'a']
['c', 'a', 'b', 'a', 'c']
['c', 'b', 'a', 'a', 'b']
['c', 'a', 'b', 'c', 'b']
['c', 'b', 'c', 'b', 'b']
['b', 'b', 'b', 'b', 'a']
['a', 'a', 'b', 'b', 'b']
['a', 'c', 'c', 'c', 'c']
['c', 'c', 'b', 'b', 'c']
['a', 'c', 'c', 'c', 'b']
['c', 'b', 'b', 'b', 'c']
['c', 'c', 'a', 'b', 'b']

Samples with repeated elements: 20


### Explanation

Because we are drawing 5 times from only 3 possible values, every sample must contain at least one repeated value. This follows from the pigeonhole principle.

## Problem 3: `choice` Repeatedly vs `choices`

Repeated calls to `choice` can produce results similar to one call to `choices`, but they are not always a drop-in replacement in code design.

### Task

1. Write `choose_loop(population, k, seed)` using repeated `choice`.
2. Write `choose_once(population, k, seed)` using `choices`.
3. Compare the two outputs for the same seed.
4. Explain why they may or may not match exactly.

### Solution

In [4]:
def choose_loop(population, k, seed=None):
    rng = random.Random(seed)
    return [rng.choice(population) for _ in range(k)]


def choose_once(population, k, seed=None):
    rng = random.Random(seed)
    return rng.choices(population, k=k)


population = [10, 20, 30, 40, 50, 60]

loop_result = choose_loop(population, 10, seed=0)
choices_result = choose_once(population, 10, seed=0)

print("Repeated choice:", loop_result)
print("choices:", choices_result)

assert len(loop_result) == len(choices_result) == 10
assert all(item in population for item in loop_result)
assert all(item in population for item in choices_result)

Repeated choice: [40, 40, 10, 30, 50, 40, 40, 30, 40, 30]
choices: [60, 50, 30, 20, 40, 30, 50, 20, 30, 40]


### Explanation

Both approaches sample with replacement. However, implementation details differ, so the exact sequence is not something you should rely on unless your code explicitly tests for a known version and behavior. Usually, test properties instead: correct length, valid elements, and reproducibility.

## Problem 4: Weighted Random Choices

Suppose a website shows one of three banners: `A`, `B`, or `C`.

The desired probabilities are:

- `A`: 70%
- `B`: 20%
- `C`: 10%

### Task

1. Use `random.choices` with weights to simulate 100,000 banner impressions.
2. Count the observed frequencies.
3. Compare the observed percentages with the expected percentages.

### Solution

In [5]:
def frequency_percentages(values):
    total = len(values)
    counts = Counter(values)
    return {
        key: {
            "count": counts[key],
            "percentage": 100 * counts[key] / total
        }
        for key in sorted(counts)
    }


rng = random.Random(42)

banners = ['A', 'B', 'C']
weights = [70, 20, 10]

impressions = rng.choices(banners, weights=weights, k=100_000)
report = frequency_percentages(impressions)

for banner, stats in report.items():
    print(banner, stats)

assert abs(report['A']['percentage'] - 70) < 1
assert abs(report['B']['percentage'] - 20) < 1
assert abs(report['C']['percentage'] - 10) < 1

A {'count': 70012, 'percentage': 70.012}
B {'count': 19912, 'percentage': 19.912}
C {'count': 10076, 'percentage': 10.076}


### Explanation

Weights are relative, not necessarily percentages. `[70, 20, 10]`, `[7, 2, 1]`, and `[0.7, 0.2, 0.1]` describe the same proportions.

## Problem 5: Equivalent Weights

The following weight lists should represent the same probability distribution:

```python
[8, 1, 1]
[80, 10, 10]
[0.8, 0.1, 0.1]
```

### Task

1. Generate 50,000 choices from `['a', 'b', 'c']` using each set of weights.
2. Compare the resulting frequencies.
3. Confirm that the percentages are close to 80%, 10%, and 10%.

### Solution

In [6]:
population = ['a', 'b', 'c']
weight_sets = [
    [8, 1, 1],
    [80, 10, 10],
    [0.8, 0.1, 0.1]
]

for weights in weight_sets:
    rng = random.Random(0)
    sample = rng.choices(population, weights=weights, k=50_000)
    percentages = frequency_percentages(sample)
    
    print("Weights:", weights)
    for item in population:
        print(item, round(percentages[item]["percentage"], 2))
    print()
    
    assert abs(percentages['a']['percentage'] - 80) < 1
    assert abs(percentages['b']['percentage'] - 10) < 1
    assert abs(percentages['c']['percentage'] - 10) < 1

Weights: [8, 1, 1]
a 80.06
b 9.97
c 9.97

Weights: [80, 10, 10]
a 80.06
b 9.97
c 9.97

Weights: [0.8, 0.1, 0.1]
a 80.06
b 9.97
c 9.97



### Explanation

Only the ratios matter. Multiplying every weight by the same positive value does not change the probability distribution.

## Problem 6: Convert Weights to Cumulative Weights

`random.choices` accepts either `weights` or `cum_weights`.

For example:

```python
weights = [8, 1, 1]
cum_weights = [8, 9, 10]
```

### Task

1. Write a function `to_cumulative(weights)`.
2. Use both `weights` and `cum_weights` to generate samples.
3. Show that they represent the same distribution.

### Solution

In [7]:
def to_cumulative(weights):
    cumulative = []
    running_total = 0
    
    for weight in weights:
        if weight < 0:
            raise ValueError("Weights must be non-negative.")
        running_total += weight
        cumulative.append(running_total)
    
    if running_total <= 0:
        raise ValueError("At least one weight must be positive.")
        
    return cumulative


weights = [8, 1, 1]
cum_weights = to_cumulative(weights)

print(cum_weights)

rng1 = random.Random(100)
rng2 = random.Random(100)

sample_with_weights = rng1.choices(['a', 'b', 'c'], weights=weights, k=20)
sample_with_cum_weights = rng2.choices(['a', 'b', 'c'], cum_weights=cum_weights, k=20)

print(sample_with_weights)
print(sample_with_cum_weights)

assert sample_with_weights == sample_with_cum_weights

[8, 9, 10]
['a', 'a', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'c', 'c', 'a', 'a', 'a', 'a', 'a', 'a', 'a']
['a', 'a', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'c', 'c', 'a', 'a', 'a', 'a', 'a', 'a', 'a']


### Explanation

Cumulative weights are running totals. They divide the interval from 0 to the total weight into regions assigned to each population item.

## Problem 7: Validate Weighted Choice Inputs

Write a safe wrapper around `random.choices`.

### Task

Create `safe_weighted_choices(population, weights, k, seed=None)` that checks:

1. `population` is not empty.
2. `weights` has the same length as `population`.
3. No weight is negative.
4. At least one weight is positive.
5. `k` is non-negative.

Then return the random choices.

### Solution

In [8]:
def safe_weighted_choices(population, weights, k, seed=None):
    if len(population) == 0:
        raise ValueError("population cannot be empty")
    
    if len(population) != len(weights):
        raise ValueError("weights must have the same length as population")
    
    if any(weight < 0 for weight in weights):
        raise ValueError("weights cannot be negative")
    
    if sum(weights) <= 0:
        raise ValueError("at least one weight must be positive")
    
    if k < 0:
        raise ValueError("k cannot be negative")
    
    rng = random.Random(seed)
    return rng.choices(population, weights=weights, k=k)


print(safe_weighted_choices(['a', 'b', 'c'], [8, 1, 1], 10, seed=0))

bad_inputs = [
    ([], [], 1),
    (['a', 'b'], [1], 1),
    (['a', 'b'], [1, -1], 1),
    (['a', 'b'], [0, 0], 1),
    (['a', 'b'], [1, 1], -1),
]

for population, weights, k in bad_inputs:
    try:
        safe_weighted_choices(population, weights, k)
    except ValueError as ex:
        print("Correctly rejected:", ex)

['b', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']
Correctly rejected: population cannot be empty
Correctly rejected: weights must have the same length as population
Correctly rejected: weights cannot be negative
Correctly rejected: at least one weight must be positive
Correctly rejected: k cannot be negative


### Explanation

Validation is important when weighted sampling is part of a larger application. Bad weights can produce confusing errors or misleading results.

## Problem 8: Empirical vs Theoretical Probability

You are given:

```python
population = ['free', 'basic', 'pro', 'enterprise']
weights = [50, 30, 15, 5]
```

### Task

1. Compute the theoretical probability of each item.
2. Generate 200,000 random choices.
3. Compute the empirical probability of each item.
4. Report the absolute error for each item.

### Solution

In [9]:
def theoretical_probabilities(population, weights):
    total_weight = sum(weights)
    return {
        item: weight / total_weight
        for item, weight in zip(population, weights)
    }


def empirical_probabilities(sample):
    total = len(sample)
    counts = Counter(sample)
    return {
        item: counts[item] / total
        for item in counts
    }


population = ['free', 'basic', 'pro', 'enterprise']
weights = [50, 30, 15, 5]

rng = random.Random(123)
sample = rng.choices(population, weights=weights, k=200_000)

theoretical = theoretical_probabilities(population, weights)
empirical = empirical_probabilities(sample)

for item in population:
    error = abs(empirical[item] - theoretical[item])
    print(
        item,
        "theoretical=", round(theoretical[item], 4),
        "empirical=", round(empirical[item], 4),
        "abs_error=", round(error, 4)
    )
    assert error < 0.01

free theoretical= 0.5 empirical= 0.5001 abs_error= 0.0001
basic theoretical= 0.3 empirical= 0.2986 abs_error= 0.0014
pro theoretical= 0.15 empirical= 0.1508 abs_error= 0.0008
enterprise theoretical= 0.05 empirical= 0.0505 abs_error= 0.0005


### Explanation

With enough samples, empirical probabilities should usually be close to theoretical probabilities. They should not be expected to match perfectly.

## Problem 9: Generate Random Integers Using `choices`

Generate 50 evenly weighted random integers between 0 and 100 inclusive.

### Task

1. Use `random.choices` with `range(101)`.
2. Confirm that every generated value is between 0 and 100.
3. Confirm that exactly 50 values are generated.
4. Make the result reproducible.

### Solution

In [10]:
def random_integers_0_to_100(k, seed=None):
    rng = random.Random(seed)
    return rng.choices(range(101), k=k)


values = random_integers_0_to_100(50, seed=2024)

print(values)

assert len(values) == 50
assert all(0 <= value <= 100 for value in values)
assert values == random_integers_0_to_100(50, seed=2024)

[47, 73, 30, 89, 41, 72, 26, 24, 82, 50, 42, 73, 97, 31, 71, 52, 73, 100, 20, 76, 47, 71, 88, 14, 21, 41, 5, 35, 42, 12, 75, 77, 39, 34, 20, 43, 31, 21, 87, 23, 4, 22, 1, 87, 85, 32, 96, 81, 42, 11]


### Explanation

`range(101)` represents integers from 0 through 100. Since no weights are provided, each integer is equally likely.

## Problem 10: Weighted Benchmark Dataset

You want to compare two styles of handling division by zero:

- LBYL: Look before you leap.
- EAFP: Easier to ask forgiveness than permission.

The denominator is expected to be zero only 10% of the time.

### Task

1. Generate one million denominators using `random.choices([0, 1], weights=[1, 9])`.
2. Benchmark an LBYL approach.
3. Benchmark an EAFP approach.
4. Print the average time per operation for each approach.
5. Explain why the result depends on how often zero occurs.

### Solution

In [11]:
def benchmark_division_handling(n=1_000_000, zero_weight=1, nonzero_weight=9, seed=0):
    rng = random.Random(seed)
    denominators = rng.choices([0, 1], weights=[zero_weight, nonzero_weight], k=n)
    
    start = perf_counter()
    for denominator in denominators:
        if denominator == 0:
            continue
        else:
            10 / denominator
    end = perf_counter()
    lbyl_time = (end - start) / n
    
    start = perf_counter()
    for denominator in denominators:
        try:
            10 / denominator
        except ZeroDivisionError:
            continue
    end = perf_counter()
    eafp_time = (end - start) / n
    
    return lbyl_time, eafp_time, Counter(denominators)


lbyl_time, eafp_time, counts = benchmark_division_handling()

print("Denominator counts:", counts)
print("LBYL average time:", lbyl_time)
print("EAFP average time:", eafp_time)

Denominator counts: Counter({1: 899923, 0: 100077})
LBYL average time: 5.909280001651496e-08
EAFP average time: 6.238809996284545e-08


### Explanation

Exceptions are relatively expensive when they happen often. If errors are rare, EAFP can be competitive or faster. If errors are common, LBYL often wins. Weighted random data lets us benchmark realistic scenarios instead of unrealistic 50/50 cases.

## Problem 11: Probability of Seeing At Least One Rare Event

Suppose a rare event has weight `1`, and a normal event has weight `99`.

So the rare event has probability `1%` on each draw.

### Task

1. Simulate 10,000 experiments.
2. In each experiment, draw 100 events using `random.choices`.
3. Count how often at least one rare event occurs.
4. Compare the simulation to the theoretical probability:

`1 - 0.99 ** 100`

### Solution

In [12]:
def rare_event_experiment(num_experiments, draws_per_experiment, seed=None):
    rng = random.Random(seed)
    found_rare = 0
    
    for _ in range(num_experiments):
        events = rng.choices(['rare', 'normal'], weights=[1, 99], k=draws_per_experiment)
        if 'rare' in events:
            found_rare += 1
            
    return found_rare / num_experiments


empirical_probability = rare_event_experiment(10_000, 100, seed=42)
theoretical_probability = 1 - 0.99 ** 100

print("Empirical probability:", empirical_probability)
print("Theoretical probability:", theoretical_probability)
print("Absolute error:", abs(empirical_probability - theoretical_probability))

assert abs(empirical_probability - theoretical_probability) < 0.03

Empirical probability: 0.632
Theoretical probability: 0.6339676587267709
Absolute error: 0.001967658726770849


### Explanation

The probability of no rare event in 100 independent draws is `0.99 ** 100`. Therefore, the probability of at least one rare event is `1 - 0.99 ** 100`.

## Problem 12: Build a Weighted Sampler Class

Create a reusable class for reproducible weighted sampling.

### Requirements

The class should:

1. Store its own random generator.
2. Accept a population and weights.
3. Validate the inputs.
4. Provide `.one()` for one random item.
5. Provide `.many(k)` for many random items.
6. Provide `.frequency_report(k)` for count and percentage summaries.
7. Be reproducible when initialized with the same seed.

### Solution

In [13]:
Freq = namedtuple("Freq", "count percentage")


class WeightedSampler:
    def __init__(self, population, weights, seed=None):
        if len(population) == 0:
            raise ValueError("population cannot be empty")
        
        if len(population) != len(weights):
            raise ValueError("population and weights must have the same length")
        
        if any(weight < 0 for weight in weights):
            raise ValueError("weights cannot be negative")
        
        if sum(weights) <= 0:
            raise ValueError("at least one weight must be positive")
        
        self.population = list(population)
        self.weights = list(weights)
        self.rng = random.Random(seed)
        
    def one(self):
        return self.rng.choices(self.population, weights=self.weights, k=1)[0]
    
    def many(self, k):
        if k < 0:
            raise ValueError("k cannot be negative")
        return self.rng.choices(self.population, weights=self.weights, k=k)
    
    def frequency_report(self, k):
        sample = self.many(k)
        counts = Counter(sample)
        return {
            item: Freq(counts[item], 100 * counts[item] / k)
            for item in self.population
        }


sampler1 = WeightedSampler(['a', 'b', 'c'], [8, 1, 1], seed=10)
sampler2 = WeightedSampler(['a', 'b', 'c'], [8, 1, 1], seed=10)

draws1 = sampler1.many(20)
draws2 = sampler2.many(20)

print(draws1)
print(draws2)

assert draws1 == draws2

sampler3 = WeightedSampler(['a', 'b', 'c'], [8, 1, 1], seed=0)
report = sampler3.frequency_report(10_000)

print(report)
assert abs(report['a'].percentage - 80) < 2

['a', 'a', 'a', 'a', 'b', 'b', 'a', 'a', 'a', 'a', 'a', 'c', 'c', 'a', 'b', 'a', 'a', 'a', 'a', 'a']
['a', 'a', 'a', 'a', 'b', 'b', 'a', 'a', 'a', 'a', 'a', 'c', 'c', 'a', 'b', 'a', 'a', 'a', 'a', 'a']
{'a': Freq(count=7994, percentage=79.94), 'b': Freq(count=1049, percentage=10.49), 'c': Freq(count=957, percentage=9.57)}


### Explanation

This class wraps best practices into a reusable object: local random state, input validation, reproducibility, and convenient reporting.

## Best-practice checklist

When using `random.choice` and `random.choices`:

1. Use `choice` for one item.
2. Use `choices` for many items with replacement.
3. Remember that `choices(..., k=n)` may return duplicates.
4. Use `weights` when some outcomes should be more likely than others.
5. Use `cum_weights` only when cumulative totals are more convenient or already available.
6. Validate that weights match the population length.
7. Remember that weights are relative, not necessarily percentages.
8. Use `Counter` for frequency analysis.
9. Use a local `random.Random(seed)` object for reproducible and isolated randomness.
10. Use weighted random data to benchmark realistic scenarios.